# Data Preprocessing and Exploratory Analysis of Functional Capacity and Quality of Life in Pediatric Patients with Marfan Syndrome
by Chloé Laignel-Granier

## Installation of required Python packages

Warning: Python and pip must be installed for this script to work.

In [ ]:
import sys
import subprocess

packages = ["pandas", "numpy", "matplotlib", "seaborn"]

for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

## Step 1: Import of the raw dataset

In [ ]:
# Load dataset
import pandas as pd
import numpy as np
file_path = "../data/Marfan.xlsx"
data = pd.read_excel(file_path)

# Total number of patients
print(f"The dataset includes {len(data)} patients.")


## Step 2 : Clean data

In [ ]:
# Rename variables and retain VAT (ventilatory threshold) for descriptive analysis of included and excluded patients
data = data.rename(columns={
    "VO2_SV1__percent_theoretical_after": "VAT_after",
    "VO2_SV1__percent_theoretical_before": "VAT_before"
})
# Standardize variable names
data.columns = data.columns.str.lower().str.strip()
data.head()

Variable names were standardized by converting all column names to lowercase and removing leading and trailing spaces to ensure consistency and prevent errors during data processing.

In [ ]:
# Data quality checks

from IPython.display import display, Markdown

display(Markdown("### Missing Data"))
missing_table = data.isna().sum().to_frame(name="Missing values")
display(missing_table)

display(Markdown("### Duplicate Observations"))
duplicates = data.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

display(Markdown("### Descriptive Statistics"))
display(data.describe().round(1))

Data quality was assessed by examining missing values, duplicate observations, and descriptive statistics. Missing data were quantified for each variable, duplicate entries were checked, and summary statistics were used to identify potential outliers and inconsistencies.

## Step 3 : Separate the analyzed and excluded patients

In [ ]:
# Define key variables for analysis
cols_all = [
    "6minwt_distance_traveled_m_before",
    "6minwt_distance_traveled_m_after",
    "pedsql_totalself_after",
    "pedsql_santeself_after",
    "pedsql_relationself_after",
    "pedsql_ecoleself_after",
    "pedsql_psycosocialself_after",  
    "pedsql_physiqueself_after"
]

# Analysable patients (no missing data)
data_included = data.dropna(subset=cols_all)

# Excluded patients
data_excluded = data[data[cols_all].isna().any(axis=1)]

# Verification
print("Total patients:", len(data))
print("Included:", len(data_included))
print("Excluded:", len(data_excluded))

Key outcome measures included the 6-minute walk test (6MWT) and PedsQL quality of life scores.

Patients with missing data for these variables at baseline or post-intervention were excluded from the primary analysis.

## Step 4 : Transformation to Long Format

In [ ]:
# Transform the data to long format for distance traveled
distance_long = pd.melt(
    data_included,
    id_vars=["id"],
    value_vars=["6minwt_distance_traveled_m_before","6minwt_distance_traveled_m_after"],
    var_name="time",
    value_name="distance"
)

# Simplify the time variable
distance_long["time"] = distance_long["time"].map({
     "6minwt_distance_traveled_m_before": "T-3",
     "6minwt_distance_traveled_m_after": "T+6"
 })
print(distance_long)

Data were converted to long format for visualization and paired analysis.

## Step 5: Export data_clean and data_excluded to R

In [ ]:
# Export data for further analysis
data_included.to_excel("../data/data_clean.xlsx", index=False)
data_excluded.to_excel("../data/data_excluded.xlsx", index=False)

The data were exported in wide format for R because this format is more suitable for paired before/after statistical analyses.

## Step 8: Analysis of 6MWT — Calculation of Changes and Data Visualization

### Calculation of change (6MWT)

In [ ]:
# Individual changes in 6MWT distance were calculated as the difference between post- and pre-intervention values.

data_included["delta_6mwt"] = data_included["6minwt_distance_traveled_m_after"] - data_included["6minwt_distance_traveled_m_before"]
print(data_included[["id", "delta_6mwt"]])

### Data visualization

#### Figure 1. Changes in 6-Minute Walk Distance Before and After Intervention

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# Global style
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12
plt.figure(figsize=(6,5))

# Long format used for grouped visualizations with Seaborn
# Boxplot showing the distribution at each timepoint
sns.boxplot(
    data=distance_long,
    x="time",
    y="distance",
    color="lightgrey"
)

# Individual data points
sns.stripplot(
    data=distance_long,
    x="time",
    y="distance",
    color="blue",
    alpha=0.5,
    jitter= False
)
# Wide format retained for paired individual trajectories
# Paired lines for each patient (before → after)
for i in range(len(data_included)):
    plt.plot(
        ["T-3", "T+6"],
        [
            data_included["6minwt_distance_traveled_m_before"].iloc[i],
            data_included["6minwt_distance_traveled_m_after"].iloc[i]
        ],
        color="grey",
        alpha=0.3,
        linewidth=1
    )

# Labels and title
plt.xlabel("Timepoint")
plt.ylabel("6-Minute Walk Distance (m)")
plt.title("Figure 1. Changes in 6-Minute Walk Distance Before and After Intervention")

# Add n = 20
plt.text(
    0.02, 0.95,
    "n = 20",
    transform=plt.gca().transAxes,
    fontsize=12,
    style="italic",
    verticalalignment="top"
)

# Save the figure
plt.savefig(
    "../figures/Figure_1_Changes_in_6MWT_Before_After.png",
    bbox_inches="tight"
)

# Display the plot
plt.show()

Data were visualized using boxplots with individual paired lines to illustrate changes in 6-minute walk distance between baseline (T−3) and post-intervention (T+6).

The results show an overall increase in walking distance at T+6, suggesting an improvement in functional capacity. Most individual trajectories follow an upward trend, although some variability remains between patients.

Overall, this figure provides a preliminary visual indication of improvement, which will be further assessed by statistical analysis.

####  Figure 2. Distribution of Changes in 6-Minute Walk Distance

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# Global style
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12
plt.figure(figsize=(6,5))

# Boxplot showing the distribution of individual changes (delta)
sns.boxplot(
    y=data_included["delta_6mwt"],
    color="lightblue"
)

# Add reference line at zero (no change)
plt.axhline(
    0,
    linestyle="--",
    color="red",
    linewidth=1,
    label="No change"
)

# Labels and title
plt.ylabel("Change in 6-Minute Walk Distance (m)")
plt.title("Figure 2. Boxplot of Changes in 6-Minute Walk Distance")

# Add n = 20
plt.text(
    0.02, 0.95,
    "n = 20",
    transform=plt.gca().transAxes,
    fontsize=12,
    style="italic",
    verticalalignment="top"
)

# Legend
plt.legend()

# Save the figure
plt.savefig(
    "../figures/Figure_2_Boxplot_of_Changes_in_6MWT.png",
    bbox_inches="tight"
)

# Display the plot
plt.show()

#### Figure_3_Histogram_of_Individual_Changes_in_the_6-Minute_Walk_Test

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
import matplotlib as mpl


# Global style
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12
plt.figure(figsize=(6,5))

# Plot
plt.figure(figsize=(6, 4))

sns.histplot(data=data_included, x="delta_6mwt", bins=10, color="lightgrey", edgecolor="black")

# Vertical line at 0
plt.axvline(x=0, linestyle="--", color="red")

# Labels and title
plt.xlabel("Change in 6-Minute Walk Distance (m)")
plt.ylabel("Number of patients")
plt.title("Figure 3. Histogram of Individual Changes in the 6-Minute Walk Test")

# Add n = 20
plt.text(
    0.02, 0.95,
    "n = 20",
    transform=plt.gca().transAxes,
    fontsize=12,
    style="italic",
    verticalalignment="top"
)

# Minimal theme
sns.despine()

# Save as PDF
plt.savefig("../figures/Figure_3_Histogram_of_Individual_Changes_in_the_6MWT.png", bbox_inches="tight")

# Show plot
plt.show()

The distribution of individual changes in 6-minute walk distance (Δ6MWT) was explored using a boxplot (Figure 2) and a histogram (Figure 3) prior to statistical analysis.

Both visualizations indicate that the majority of patients experienced an improvement, as reflected by a median above zero and a predominance of positive values. However, some variability is observed, with a few patients showing no improvement or a decrease in performance.

The histogram further suggests a right-skewed distribution, indicating that some patients achieved larger improvements.

Overall, these exploratory analyses support the use of a non-parametric statistical approach.

#### Table 2. Distribution and changes in 6-minute walk distance

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib as mpl

# Global style
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12
plt.figure(figsize=(6,5))

# Format median[IQR]
summary_table = distance_long.groupby("time")["distance"].agg(
    Median="median",
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75)
)

# Create formatted column
summary_table["Median (IQR)"] = (
    summary_table["Median"].round(1).astype(str) + "(" +
    summary_table["Q1"].round(1).astype(str) + "–" +
    summary_table["Q3"].round(1).astype(str) + ")"
)

# Keep only useful column
summary_table = summary_table[["Median (IQR)"]]

# Rename index
summary_table.index = ["T-3", "T+6"]

# delta median(IQR)
delta_median = data_included["delta_6mwt"].median()
delta_q1 = data_included["delta_6mwt"].quantile(0.25)
delta_q3 = data_included["delta_6mwt"].quantile(0.75)

delta_summary = pd.DataFrame({
    "Median (IQR)": [
        f"{round(delta_median,1)} ({round(delta_q1,1)}–{round(delta_q3,1)})"
    ]
}, index=["Δ (T+6 – T-3)"])

# Delta summary 
improvement = (data_included["delta_6mwt"] > 0).sum()
no_change = (data_included["delta_6mwt"] == 0).sum()
decrease = (data_included["delta_6mwt"] < 0).sum()

total = len(data_included)
delta_table = pd.DataFrame({
    "Change (n, %)": [
        f"{improvement} ({round(improvement/total*100,1)}%)",
        f"{no_change} ({round(no_change/total*100,1)}%)",
        f"{decrease} ({round(decrease/total*100,1)}%)"
    ]
}, index=["Improved", "No change", "Decreased"])


# Final table
final_table = pd.concat([summary_table, delta_summary, delta_table])


# Export PDF
fig, ax = plt.subplots(figsize=(6,4))
ax.axis('off')

table = ax.table(
    cellText=final_table.values,
    colLabels=final_table.columns,
    rowLabels=final_table.index,
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(12)

for key, cell in table.get_celld().items():
    cell.set_text_props(fontfamily='Times New Roman')
    
# Title
plt.title(
    "Table 2. Distribution and changes in 6-minute walk distance (n = 20)",
    fontsize=12,
    pad=20
)

plt.savefig("../figures/Table_2_Distribution_and_changes_in_6MWT.png", bbox_inches="tight")
plt.close()

from IPython.display import display

display(final_table.style.set_caption(
    "Table 2. Distribution and changes in 6-minute walk distance (n = 20)"
))

Descriptive statistics are reported as median [interquartile range].

Although the median 6-minute walk distance slightly decreased between baseline and post-intervention, the median individual change was positive (+57.0 m), suggesting an overall improvement with inter-individual variability.

Overall, 60% of patients improved, while 40% showed a decrease.

Exploratory analysis suggests a general trend toward improvement in functional capacity after the intervention. This observation will be further examined using statistical tests.